# 🧥 ITLALA — Clothing Classifier API

This notebook runs a **Qwen2.5-VL-7B** vision-language model to classify clothing items.

**Pipeline:**
1. Receive clothing image via API
2. Preprocess: Remove background using `rembg`, place on white background
3. Classify: Use Qwen2.5-VL to analyze and categorize the clothing
4. Return: JSON metadata + processed image (base64)

---

### Prerequisites
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Add your tokens in **Secrets** (left sidebar 🔑):
   - `HF_TOKEN` — Hugging Face access token
   - `NGROK_TOKEN` — ngrok authtoken

Then run all cells in order.

## Cell 1 — Install Dependencies

In [7]:
# Cell 1: Install all required packages
!pip install -q numpy==2.1.3  # Fix numpy compatibility
!pip install -q transformers accelerate
!pip install -q qwen-vl-utils
!pip install -q fastapi nest-asyncio pyngrok uvicorn python-multipart
!pip install -q "rembg[gpu]" pillow onnxruntime-gpu
!pip install -q bitsandbytes
!pip install -q huggingface_hub[hf_transfer]

print("\n✅ All packages installed successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rembg 2.0.76 requires numpy<3.0.0,>=2.3.0, but you have numpy 2.1.3 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numb

## Cell 2 — Authentication & Configuration

In [8]:
# ============================================================
# Cell 2: Set up authentication tokens
# ============================================================

import os
from google.colab import userdata

# Load tokens from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["NGROK_TOKEN"] = userdata.get("NGROK_TOKEN")

print("✅ Tokens loaded from Colab Secrets")
print(f"   HF_TOKEN: {'*' * 8}...{os.environ['HF_TOKEN'][-4:]}")
print(f"   NGROK_TOKEN: {'*' * 8}...{os.environ['NGROK_TOKEN'][-4:]}")

✅ Tokens loaded from Colab Secrets
   HF_TOKEN: ********...NQqi
   NGROK_TOKEN: ********...6NC1


## Cell 3 — Load Qwen2.5-VL Model

In [9]:
# ============================================================
# Cell 3: Load the Qwen2_5_VL model with 4-bit quantization
# ============================================================

import os
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"📦 Loading model: {MODEL_ID}")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 1. تعريف إعدادات الـ Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 2. تحميل الموديل باستخدام الإعدادات الجديدة
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quantization_config,
    token=os.environ["HF_TOKEN"]
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"]
)

print(f"\n✅ Model loaded successfully!")
print(f"   Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

📦 Loading model: Qwen/Qwen2.5-VL-7B-Instruct
   GPU: Tesla T4
   VRAM: 15.6 GB


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

INFO:     102.46.52.56:0 - "OPTIONS /api/classify HTTP/1.1" 200 OK

📸 New classification request: file_00000000090c7243a7e963097880b0fb.png
   Content type: image/png
   File size: 2014.4 KB

🔄 Step 1: Preprocessing...
   📐 Original size: (1448, 1086)
   🧹 Removing background...
   📐 Processed size: (1024, 768)
   ✅ Preprocessing complete

🔄 Step 2: Classifying...
   🤖 Running Qwen classification...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   📝 Raw response: {
    "category": "top",
    "subcategory": "graphic t-shirt",
    "gender": "unisex",
    "material": "cotton",
    "pattern": "solid",
    "fit": "oversized",
    "sleeve": "short sleeves",
    "col...
   ✅ Classification complete

✅ Done in 20.6s
   Category: top
   Subcategory: graphic t-shirt
   Colors: ['maroon']

INFO:     102.46.52.56:0 - "POST /api/classify HTTP/1.1" 200 OK

✅ Model loaded successfully!
   Memory used: 10.31 GB


## Cell 4 — Image Preprocessing Functions

In [10]:
# ============================================================
# Cell 4: Image preprocessing — background removal + cleanup
# ============================================================

from rembg import remove
from PIL import Image
import io
import base64

def preprocess_clothing_image(image_bytes: bytes):
    """
    Preprocess a clothing image:
    1. Remove background using rembg (U2-Net)
    2. Place clothing on clean white background
    3. Resize to consistent dimensions (max 1024px)
    4. Return PIL Image + base64 string
    """
    # Open the raw image
    input_image = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
    print(f"   📐 Original size: {input_image.size}")

    # Remove background (returns RGBA with transparent bg)
    print("   🧹 Removing background...")
    no_bg = remove(input_image)

    # Create white background and composite
    white_bg = Image.new("RGBA", no_bg.size, (255, 255, 255, 255))
    final = Image.alpha_composite(white_bg, no_bg).convert("RGB")

    # Resize for consistency (keep aspect ratio, max 1024px)
    final.thumbnail((1024, 1024), Image.LANCZOS)
    print(f"   📐 Processed size: {final.size}")

    # Convert to base64
    buffer = io.BytesIO()
    final.save(buffer, format="JPEG", quality=90)
    img_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    print("   ✅ Preprocessing complete")
    return final, img_b64


# Quick test with a blank image
test_img = Image.new("RGB", (100, 100), (200, 200, 200))
buf = io.BytesIO()
test_img.save(buf, format="PNG")
_, test_b64 = preprocess_clothing_image(buf.getvalue())
print(f"\n✅ Preprocessing pipeline ready! (test base64 length: {len(test_b64)})")

   📐 Original size: (100, 100)
   🧹 Removing background...
   📐 Processed size: (100, 100)
   ✅ Preprocessing complete

✅ Preprocessing pipeline ready! (test base64 length: 2444)


## Cell 5 — Clothing Classification Function

In [11]:
# ============================================================
# Cell 5: Qwen VL classification — analyze clothing items
# ============================================================

import json
import re

CLASSIFICATION_PROMPT = """You are an expert fashion analyst. Analyze this clothing item image very carefully.

Return ONLY a valid JSON object with these exact fields (no extra text before or after):

{
    "category": "<one of: top, bottom, outerwear, footwear, accessory, dress, activewear>",
    "subcategory": "<specific type, e.g.: graphic t-shirt, slim jeans, bomber jacket, sneakers, etc.>",
    "gender": "<one of: men, women, unisex>",
    "material": "<detected material, e.g.: cotton, denim, polyester, leather, wool, silk, linen>",
    "pattern": "<detected pattern, e.g.: solid, striped, plaid, polka dot, floral, graphic, abstract, checkered>",
    "fit": "<fit type, e.g.: regular, slim, oversized, relaxed, tailored, loose>",
    "sleeve": "<sleeve type, e.g.: short sleeves, long sleeves, sleeveless, 3/4 sleeves, or N/A if not applicable>",
    "colors": ["<list of main colors detected>"],
    "season": ["<list of suitable seasons from: spring, summer, fall, winter>"],
    "description": "<A natural, detailed description of the clothing item in 1-2 sentences>"
}

IMPORTANT: Return ONLY the JSON object. No markdown, no code blocks, no explanation."""


def classify_clothing(image: Image.Image) -> dict:
    """
    Use Qwen VL to analyze a clothing image and return structured metadata.
    """
    print("   🤖 Running Qwen classification...")

    # Build the conversation
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": CLASSIFICATION_PROMPT}
            ]
        }
    ]

    # Process inputs
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate classification
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )

    # Decode response
    response_text = processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )[0]

    print(f"   📝 Raw response: {response_text[:200]}...")

    # Extract JSON from response
    json_match = re.search(r'\{[\s\S]*\}', response_text)
    if not json_match:
        raise ValueError(f"No JSON found in model response: {response_text[:300]}")

    classification = json.loads(json_match.group())

    # Validate required fields
    required_fields = ["category", "subcategory", "gender", "material",
                       "pattern", "fit", "sleeve", "colors", "season", "description"]
    for field in required_fields:
        if field not in classification:
            classification[field] = "unknown" if field not in ["colors", "season"] else ["unknown"]

    # Ensure colors and season are arrays
    if isinstance(classification.get("colors"), str):
        classification["colors"] = [classification["colors"]]
    if isinstance(classification.get("season"), str):
        classification["season"] = [classification["season"]]

    print("   ✅ Classification complete")
    return classification


print("✅ Classification function ready!")

✅ Classification function ready!


## Cell 6 — FastAPI Server + ngrok Tunnel

⚠️ **This cell runs indefinitely** — it starts the API server.

After running, copy the **ngrok URL** and paste it in your `.env` file as `VITE_CLASSIFIER_API_URL`.

In [12]:
# ============================================================
# Cell 6: FastAPI server with ngrok tunnel
# ============================================================

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import uuid
import traceback
import time
import threading

# Create FastAPI app
app = FastAPI(
    title="ITLALA Clothing Classifier",
    description="AI-powered clothing classification using Qwen2.5-VL",
    version="1.0.0"
)

# Enable CORS for frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/api/health")
async def health_check():
    """Check if the API and model are ready"""
    return {
        "status": "ok",
        "model": MODEL_ID,
        "gpu": torch.cuda.get_device_name(0),
        "vram_used_gb": round(torch.cuda.memory_allocated() / 1e9, 2)
    }


@app.post("/api/classify")
async def classify_endpoint(file: UploadFile = File(...)):
    """
    Receive a clothing image, preprocess it, classify it,
    and return structured JSON + processed image.
    """
    start_time = time.time()

    # Validate file type
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type: {file.content_type}. Please upload an image."
        )

    try:
        print(f"\n{'='*50}")
        print(f"📸 New classification request: {file.filename}")
        print(f"   Content type: {file.content_type}")

        # Read image bytes
        image_bytes = await file.read()
        print(f"   File size: {len(image_bytes) / 1024:.1f} KB")

        # Step 1: Preprocess (remove background + white bg)
        print("\n🔄 Step 1: Preprocessing...")
        processed_image, processed_b64 = preprocess_clothing_image(image_bytes)

        # Step 2: Classify with Qwen
        print("\n🔄 Step 2: Classifying...")
        classification = classify_clothing(processed_image)

        # Step 3: Build response
        item_id = f"custom_{uuid.uuid4().hex[:12]}"
        classification["id"] = item_id
        classification["imageBase64"] = f"data:image/jpeg;base64,{processed_b64}"
        classification["source"] = "user_upload"

        elapsed = time.time() - start_time
        print(f"\n✅ Done in {elapsed:.1f}s")
        print(f"   Category: {classification['category']}")
        print(f"   Subcategory: {classification['subcategory']}")
        print(f"   Colors: {classification['colors']}")
        print(f"{'='*50}\n")

        return JSONResponse(content={
            "success": True,
            "data": classification,
            "processing_time": round(elapsed, 2)
        })

    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
        raise HTTPException(
            status_code=500,
            detail=f"Failed to parse model output as JSON: {str(e)}"
        )
    except Exception as e:
        print(f"❌ Error: {e}")
        traceback.print_exc()
        raise HTTPException(
            status_code=500,
            detail=f"Classification error: {str(e)}"
        )


# ─────────────────────────────────────────────────────────────
# Start the server with ngrok tunnel
# ─────────────────────────────────────────────────────────────

nest_asyncio.apply()

# Set up ngrok
ngrok.set_auth_token(os.environ["NGROK_TOKEN"])
public_url = ngrok.connect(8000).public_url

print("\n" + "=" * 60)
print("🚀 ITLALA Clothing Classifier API is LIVE!")
print("=" * 60)
print(f"")
print(f"   🌐 Public URL:  {public_url}")
print(f"   ❤️  Health:     {public_url}/api/health")
print(f"   📸 Classify:    POST {public_url}/api/classify")
print(f"")
print(f"   📋 Copy this to your .env file:")
print(f"   VITE_CLASSIFIER_API_URL={public_url}")
print(f"")
print("=" * 60)
print("\n⏳ Server running... (keep this cell running)\n")

# Run the server (blocks until interrupted)
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
# تشغيل السيرفر في الخلفية
thread = threading.Thread(target=server.run)
thread.start()


🚀 ITLALA Clothing Classifier API is LIVE!

   🌐 Public URL:  https://disallow-drab-eggshell.ngrok-free.dev
   ❤️  Health:     https://disallow-drab-eggshell.ngrok-free.dev/api/health
   📸 Classify:    POST https://disallow-drab-eggshell.ngrok-free.dev/api/classify

   📋 Copy this to your .env file:
   VITE_CLASSIFIER_API_URL=https://disallow-drab-eggshell.ngrok-free.dev


⏳ Server running... (keep this cell running)



INFO:     Started server process [11662]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
